# Analyze Source Data

Source is delivered in /Documents/ as pdf-Files. Metadata from Documents is delivered as .csv in /Documents/Documentlist.csv

Text is extractet from pdf and stored in npy. Parts of Analyze is done on extractet text

In [ ]:


import os
import sys
import argparse
import numpy as np

# Load the data from the .npy file
data = np.load(r'Documents\text.npy', allow_pickle=True)

# Analyze the data
print("Type of data:", type(data))
print("Shape of data:", getattr(data, 'shape', 'No shape attribute'))
print("First 5 entries:", data[:5])


In [ ]:
import pandas as pd

# Load the metadata from the CSV file, skipping bad lines
documentlist_df = pd.read_csv(r'Documents/Documentlist.csv', on_bad_lines='skip', sep=';')

# Display basic information about the dataframe
print(documentlist_df.info())
print(documentlist_df.head())


In [ ]:


documentlist_df.loc[documentlist_df['Titel'].str.contains('Reglement', case=False, na=False), 'Typ'] = 'Reglement'

# Count the occurrences of each 'Typ'
typ_counts = documentlist_df['Typ'].value_counts()
print(typ_counts)

# Create a DataFrame from the numpy array 'data'
data_df = pd.DataFrame(data)
# Expand the dictionary in each row into separate columns
data_df = pd.json_normalize(data_df[0])
extracted_columns = data_df.columns.tolist()
print("Columns in data_df:", extracted_columns)
# Ensure the 'filename' column exists in data_df

# Remove file extension and possible page numbers from 'filename' in data_df for matching
data_df['base_filename'] = data_df['filename'].str.extract(r'^(.*?)(?:\.pdf)?(?:-\d+)?\.txt$')[0].str.strip()

# Remove file extension from 'Titel' in documentlist_df for matching
documentlist_df['base_titel'] = documentlist_df['Titel']

# Merge the dataframes on the cleaned filename
merged_df = pd.merge(documentlist_df, data_df, left_on='base_titel', right_on='base_filename', how='right')

# Show a preview of the merged dataframe
print(merged_df.head())

merged_df['text_len'] = merged_df['text'].apply(lambda x: len(x) if isinstance(x, str) else 0)
print(merged_df.info())
print(merged_df[['Titel', 'text_len']].head())


plot_columns = ['Titel', 'text_len']
import matplotlib.pyplot as plt
# Plotting the text length distribution
plt.figure(figsize=(10, 6))
merged_df['text_len'].hist(bins=50)
plt.title('Text Length Distribution')
plt.xlabel('Text Length')
plt.ylabel('Frequency')


text_len_per_typ = merged_df.groupby('Typ')['text_len'].sum()
print(text_len_per_typ)
